In [17]:
import pandas as pd
from sklearn.model_selection import (
    train_test_split, GridSearchCV, StratifiedKFold
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.ensemble import VotingClassifier

In [18]:
df = pd.read_csv('datasets/adult_cleaned.csv')
df.head()

,age,workclass,fnlwgt,education,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,target
0,39,State-gov,77516,Bachelors,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,1
1,50,Self-emp-not-inc,83311,Bachelors,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,1
2,38,Private,215646,HS-grad,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,1
3,53,Private,234721,11th,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,1
4,28,Private,338409,Bachelors,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,1


In [19]:
X = df.drop('target', axis=1)
y = df['target']

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [21]:
numeric_features = [
    'age', 'fnlwgt', 'capital-gain',
    'capital-loss', 'hours-per-week'
]
categorical_features = [
    'workclass', 'education', 'marital-status', 'occupation',
    'relationship', 'race', 'sex', 'native-country'
]

In [22]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [23]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [24]:
models = {
    'LogisticRegression': {
        'model': LogisticRegression(random_state=42),
        'params': {}
    },
    'DecisionTree': {
        'model': DecisionTreeClassifier(random_state=42),
        'params': {
            'classifier__max_depth': [10, 20, None],
            'classifier__criterion': ['gini', 'entropy'],
        }
    },
    'KNN': {
        'model': KNeighborsClassifier(),
        'params': {
            'classifier__n_neighbors': [3, 5, 10, 15],
        }
    },
    'NaiveBayes': {
        'model': GaussianNB(),
        'params': {}
    }
}

In [25]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_models = {}
results = []

for name, config in models.items():
    print(f"\n{'='*60}")
    print(f"Training {name}")
    print(f"{'='*60}")

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', config['model'])
    ])

    grid_search = GridSearchCV(
        pipeline,
        config['params'],
        cv=cv,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X_train, y_train)

    best_models[name] = grid_search.best_estimator_

    y_pred = grid_search.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_pred)

    print(f"\nBest parameters: {grid_search.best_params_}")
    print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")
    print(f"Test accuracy: {test_accuracy:.4f}")

    results.append({
        'model': name,
        'best_cv_score': grid_search.best_score_,
        'test_score': test_accuracy,
        'best_params': grid_search.best_params_
    })

    print("\nClassification Report on Test Set:")
    print(classification_report(y_test, y_pred))


Training LogisticRegression
Fitting 5 folds for each of 1 candidates, totalling 5 fits

Best parameters: {}
Best cross-validation accuracy: 0.8478
Test accuracy: 0.8503

Classification Report on Test Set:
              precision    recall  f1-score   support

           0       0.74      0.60      0.67      2242
           1       0.88      0.93      0.90      6803

    accuracy                           0.85      9045
   macro avg       0.81      0.77      0.78      9045
weighted avg       0.84      0.85      0.84      9045


Training DecisionTree
Fitting 5 folds for each of 6 candidates, totalling 30 fits

Best parameters: {'classifier__criterion': 'entropy', 'classifier__max_depth': 10}
Best cross-validation accuracy: 0.8528
Test accuracy: 0.8549

Classification Report on Test Set:
              precision    recall  f1-score   support

           0       0.77      0.59      0.67      2242
           1       0.88      0.94      0.91      6803

    accuracy                           

In [26]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('best_cv_score', ascending=False)

results_df

,model,best_cv_score,test_score,best_params
1,DecisionTree,0.852807,0.854947,"{'classifier__criterion': 'entropy', 'classifi..."
0,LogisticRegression,0.847804,0.850304,{}
2,KNN,0.837742,0.837922,{'classifier__n_neighbors': 15}
3,NaiveBayes,0.622688,0.644887,{}


In [27]:
best_model_name = results_df.iloc[0]['model']
best_model = best_models[best_model_name]
print(f"Best Model: {best_model_name}")
print(f"{'='*60}")
print(f"Best cross-validation accuracy: {results_df.iloc[0]['best_cv_score']:.4f}")
print(f"Test accuracy: {results_df.iloc[0]['test_score']:.4f}")
print(f"Best parameters: {results_df.iloc[0]['best_params']}")

Best Model: DecisionTree
Best cross-validation accuracy: 0.8528
Test accuracy: 0.8549
Best parameters: {'classifier__criterion': 'entropy', 'classifier__max_depth': 10}


In [28]:
best_estimators = [(name, model) for name, model in best_models.items()]
best_estimators.pop() # Remove NaiveBayes

('NaiveBayes',
 Pipeline(steps=[('preprocessor',
                  ColumnTransformer(transformers=[('num',
                                                   Pipeline(steps=[('imputer',
                                                                    SimpleImputer(strategy='median')),
                                                                   ('scaler',
                                                                    StandardScaler())]),
                                                   ['age', 'fnlwgt',
                                                    'capital-gain',
                                                    'capital-loss',
                                                    'hours-per-week']),
                                                  ('cat',
                                                   Pipeline(steps=[('imputer',
                                                                    SimpleImputer(strategy='most_frequent')),
                    

In [29]:
voting_hard = VotingClassifier(
    estimators=best_estimators,
    voting='hard'
)
voting_hard.fit(X_train, y_train)
y_pred_hard = voting_hard.predict(X_test)

In [30]:
accuracy_score(y_test, y_pred_hard)

0.8545052515201769

In [31]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

model = Sequential([
    Dense(
        64, activation='relu',
        input_shape=(X_train_processed.shape[1],)
    ),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])
model.compile(
    optimizer='adam', loss='binary_crossentropy',
    metrics=['accuracy']
)
early_stop = EarlyStopping(patience=5, restore_best_weights=True)
model.fit(
    X_train_processed, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)
ann_y_pred = (model.predict(X_test_processed) > 0.5).astype(int)
ann_acc = accuracy_score(y_test, ann_y_pred)

Epoch 1/50
453/453 [==============================] - 3s 5ms/step - loss: 0.3791 - accuracy: 0.8204 - val_loss: 0.3359 - val_accuracy: 0.8449
Epoch 2/50
453/453 [==============================] - 2s 4ms/step - loss: 0.3344 - accuracy: 0.8451 - val_loss: 0.3227 - val_accuracy: 0.8499
Epoch 3/50
453/453 [==============================] - 2s 4ms/step - loss: 0.3277 - accuracy: 0.8483 - val_loss: 0.3153 - val_accuracy: 0.8553
Epoch 4/50
453/453 [==============================] - 2s 4ms/step - loss: 0.3233 - accuracy: 0.8497 - val_loss: 0.3171 - val_accuracy: 0.8521
Epoch 5/50
453/453 [==============================] - 2s 4ms/step - loss: 0.3213 - accuracy: 0.8529 - val_loss: 0.3196 - val_accuracy: 0.8520
Epoch 6/50
453/453 [==============================] - 2s 4ms/step - loss: 0.3186 - accuracy: 0.8548 - val_loss: 0.3139 - val_accuracy: 0.8530
Epoch 7/50
453/453 [==============================] - 2s 4ms/step - loss: 0.3161 - accuracy: 0.8554 - val_loss: 0.3145 - val_accuracy: 0.8532
Epoch 

In [32]:
print('Neural Network Accuracy: ', ann_acc)

Neural Network Accuracy:  0.8532891100055279
